<a href="https://colab.research.google.com/github/kmeng01/rome/blob/main/notebooks/rome.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" align="left"/></a>&nbsp;or in a local notebook.

# Multi- Rank-One Model Editing (Multi-ROME)
This notebook enables interactive experimentation with Multi-ROME and several other comparable baselines.
The goal is to write new facts (e.g. counterfactuals) into existing pre-trained models with generalization and specificity, where a fact is edited into multiple layers as multiple copies.

In [3]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
if os.path.basename(os.getcwd()) == "KE4MHQ":
    os.chdir("rome")
!ls


import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from util import nethook
from util.generate import generate_interactive, generate_fast

from experiments.py.demo import demo_model_editing, stop_execution
import json
import time

from util.eval_greedy import eval_editing


import numpy as np
import random

def set_seed(seed=42):
    random.seed(seed)  # Python random module
    np.random.seed(seed)  # NumPy
    torch.manual_seed(seed)  # PyTorch CPU
    torch.cuda.manual_seed(seed)  # PyTorch GPU
    torch.cuda.manual_seed_all(seed)  # Multi-GPU
    torch.backends.cudnn.deterministic = True  # Ensure deterministic behavior
    torch.backends.cudnn.benchmark = False  # Disable auto-optimization

set_seed(42)

Here, you can specify a GPT model (`MODEL_NAME`).

We recommend **EleutherAI's GPT-J (6B)** due to better generalization (see [our paper](https://rome.baulab.info/) for details), but GPT-2 XL (1.5B) consumes less memory.
* `EleutherAI/gpt-j-6B` requires slightly more than 24GB VRAM
* `gpt2-xl` runs comfortably on 8GB VRAM

In [ ]:
IS_COLAB = False
ALL_DEPS = False
try:
    import google.colab, torch, os

    IS_COLAB = True
    os.chdir("/content/rome")
    if not torch.cuda.is_available():
        raise Exception("Change runtime type to include a GPU.")
except ModuleNotFoundError as _:
    pass


# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = 'cpu'
device = torch.device('cuda:3')
print(f"Using device: {device}")

ALG_NAME = "ROME-Multi" # alternatively: "ROME"
MODEL_NAME = "EleutherAI/gpt-j-6B" # alternatively: "gpt2-xl"

json_path_dict = {
    "gpt2-xl":"hparams/ROME/gpt2-xl.json",
    "EleutherAI/gpt-j-6B":"hparams/ROME/EleutherAI_gpt-j-6B.json"
}



Using device: cuda:3


In [ ]:

model, tok = (
    AutoModelForCausalLM.from_pretrained(MODEL_NAME, low_cpu_mem_usage=IS_COLAB).to(
        device
    ),
    AutoTokenizer.from_pretrained(MODEL_NAME),
)
tok.pad_token = tok.eos_token
model.config

#  Pipeline for testing multiple insertions on MQuake

In [11]:



json_path_dict = {
    "gpt2-xl":"hparams/ROME/gpt2-xl.json",
    "EleutherAI/gpt-j-6B":"hparams/ROME/EleutherAI_gpt-j-6B.json"
}



# context used when testing the editted model
context_file = "dsets/rel-prompts.json"
with open(context_file, "r") as f:
    rel_prompts = json.load(f)



def test_multi_rome(layers_to_edit, edit_hop, continue_from=0, max_cases=1e9, save_dir=None):

    with open(json_path_dict[MODEL_NAME], "r") as f:
        data = json.load(f)
    data["layers"] = layers_to_edit
    with open(json_path_dict[MODEL_NAME], "w") as f:
        json.dump(data, f, indent=2)

    ds_file = "dsets/ds_classification/"+edit_hop+"_edits.json"
    with open(ds_file, "r") as f:
        mhq_ds = json.load(f)

    print("Start time: ", time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()))
    print("layers_to_edit: ", layers_to_edit)
    print("ds_file: ", ds_file)
    correct = 0
    count = 0
    for i in range(len(mhq_ds)):
    # for i in range(100):
        if i < continue_from:
            continue
        if i >= continue_from + max_cases:
            break
        case = mhq_ds[i]
        request = case["requested_rewrite"]
        generation_prompts = []

        print("\n\n"+4*"***********************************************")
        print(f"Request {i+1}, case_id: {case['case_id']}")


        # Restore fresh copy of model
        try:
            with torch.no_grad():
                for k, v in orig_weights.items():
                    nethook.get_parameter(model, k)[...] = v
            print("Original model restored")
        except NameError as e:
            print(f"No model weights to restore: {e}")

        # Execute rewrite
        model_new, orig_weights = demo_model_editing(
            model, tok, request, generation_prompts, alg_name=ALG_NAME,generate_prompts=False 
            )

        if save_dir is None:
            # use default save_dir
            save_dir = edit_hop+"-Eval-" + "-".join(map(str, layers_to_edit))

        if eval_editing(model, case, rel_prompts, tok, save_dir=save_dir):
            correct += 1

        count+=1

        print(f"Correct: {correct}/{count}")

    print("End time: ", time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()))

## Test multi-edit on a single case


In [ ]:

case_index = 66
hop_to_edit = "hop1" # or "hop2"
test_multi_rome(layers_to_edit=[[5,10,15,20]], edit_hop=hop_to_edit, continue_from=case_index, max_cases=1, save_dir="my_test")

# case_index = 33
# test_multi_rome(layers_to_edit=[[5,15],[10,20]], edit_hop="both", continue_from=case_index, max_cases=1, save_dir="my_test")

## sweep over different settings

result folder will be saved under KE4MHQ/rome/multi-edit-results

In [ ]:
layers_sweep = [
    [[5,10,15,20]],
    [[5]]

]
hop_sweep = [
    "hop1", 
    "hop2",
    # "both"
]

for layers in layers_sweep:
    for hop in hop_sweep:
        test_multi_rome(layers_to_edit=layers, edit_hop=hop, continue_from=0, max_cases=100)

## Count the accuracy for given folders

In [ ]:
import os
import json

# folder_dir = "hop2-Eval-5-10-15-20"
# folder_dir ="both-Eval-[5, 15]-[10, 20]"
# folder_dir = "both-Eval-[5]-[5]"
# folder_dir = "hop2-Eval-[10]"
# folder_dir = "hop2-Eval-[5]"
# folder_dir = "hop1-Eval-[5, 10, 15, 20]"
# folder_dir = "hop2-Eval-5-10-15-20"

folder_dir_sweep = ["hop1-Eval-[5]", "hop1-Eval-[10]", "hop1-Eval-[15]", "hop1-Eval-[20]", "hop2-Eval-[5]", "hop2-Eval-[10]", "hop2-Eval-[15]", "hop2-Eval-[20]"]
def check_correctness(folder_dir):
    correct = 0
    total = len(os.listdir(folder_dir))
    # iterate over the json files in the folder
    for filename in os.listdir(folder_dir):
        if filename.endswith(".json"):
            with open(os.path.join(folder_dir, filename), "r") as f:
                results = json.load(f)

                # results[-2]["new_answer_alias"] = data["new_answer_alias"]
                # check correctness of the MHQ answer
                results[-3]["correct"] = results[-3]["model_responses"]["greedy"].lstrip().\
                                        startswith((results[-3]["answer"]))
                if results[-3]["correct"]:
                    correct += 1
    print(f"{folder_dir} Correctness: {correct}/{total}")

for folder_dir in folder_dir_sweep:
    check_correctness(folder_dir)

## Evaluation for language metrics

In [ ]:
! python3 -m experiments.summarize --dir_name=ROME-Multi --runs=run_003


## compute covariance matrix for gpt-j layer>20

In [ ]:
import os
from pathlib import Path

import torch
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

from util.globals import *
from util.nethook import Trace, set_requires_grad
from util.runningstats import CombinedStat, Mean, NormMean, SecondMoment, tally
raw_ds = load_dataset(
            "wikipedia",
            dict(wikitext="wikitext-103-raw-v1", wikipedia="20200501.en", cache_dir='./wiki-data/')["wikipedia"],
            cache_dir='./wiki-data'
        )